# Ekstraksi ROI Mata dengan MediaPipe (Offline)

# Tujuan:
Membaca dataset NTHUDDD2 (full‑face).

Menjalankan MediaPipe FaceMesh sekali saja per gambar.

Memotong ROI mata; jika gagal, simpan full‑frame sebagai fallback.

Menyimpan hasil ke folder baru Dataset_nthuddd2_roi dengan struktur sama (train/val/test, drowsy/notdrowsy).

Mencatat file yang rusak atau gagal deteksi ke log .txt.

# Code: setup & MediaPipe

In [1]:
import os
import cv2
from tqdm import tqdm
import mediapipe as mp

RAW_ROOT = r"C:\kuliah-sementara\SKRIPSI\Dataset_nthuddd2"
ROI_ROOT = r"C:\kuliah-sementara\SKRIPSI\Dataset_nthuddd2_roi"

SPLITS = ["train", "val", "test"]
CLASSES = ["drowsy", "notdrowsy"]

os.makedirs(ROI_ROOT, exist_ok=True)

mp_face_mesh = mp.solutions.face_mesh
face_mesh = mp_face_mesh.FaceMesh(
    static_image_mode=True,
    max_num_faces=1,
    refine_landmarks=True,
    min_detection_confidence=0.5,
)

LEFT_EYE_IDX  = [33, 133, 160, 159, 158, 144]
RIGHT_EYE_IDX = [362, 263, 387, 386, 385, 373]


# Code: fungsi detect_eye_roi

In [2]:
def detect_eye_roi(bgr_img, expand_ratio=0.3):
    h, w, _ = bgr_img.shape
    rgb = cv2.cvtColor(bgr_img, cv2.COLOR_BGR2RGB)
    results = face_mesh.process(rgb)

    if not results.multi_face_landmarks:
        return None

    face_landmarks = results.multi_face_landmarks[0]

    xs, ys = [], []
    for idx_list in [LEFT_EYE_IDX, RIGHT_EYE_IDX]:
        for idx in idx_list:
            lm = face_landmarks.landmark[idx]
            xs.append(int(lm.x * w))
            ys.append(int(lm.y * h))

    if not xs:
        return None

    x_min, x_max = min(xs), max(xs)
    y_min, y_max = min(ys), max(ys)

    x_margin = int((x_max - x_min) * expand_ratio)
    y_margin = int((y_max - y_min) * expand_ratio)

    x_min = max(0, x_min - x_margin)
    x_max = min(w, x_max + x_margin)
    y_min = max(0, y_min - y_margin)
    y_max = min(h, y_max + y_margin)

    if x_max <= x_min or y_max <= y_min:
        return None

    roi = bgr_img[y_min:y_max, x_min:x_max]
    return roi


#  Code: loop ekstraksi ROI + logging

In [3]:
corrupted_files = []
fallback_fullframe = []

for split in SPLITS:
    for cls in CLASSES:
        src_dir = os.path.join(RAW_ROOT, split, cls)
        dst_dir = os.path.join(ROI_ROOT, split, cls)
        os.makedirs(dst_dir, exist_ok=True)

        files = [f for f in os.listdir(src_dir)
                 if f.lower().endswith((".jpg", ".jpeg", ".png"))]

        for fname in tqdm(files, desc=f"{split}/{cls}"):
            src_path = os.path.join(src_dir, fname)
            dst_path = os.path.join(dst_dir, fname)

            img_bgr = cv2.imread(src_path)
            if img_bgr is None:
                corrupted_files.append(src_path)
                continue

            eye_roi = detect_eye_roi(img_bgr)

            if eye_roi is None or eye_roi.shape[0] < 20 or eye_roi.shape[1] < 20:
                # fallback: simpan full-frame
                roi = img_bgr
                fallback_fullframe.append(src_path)
            else:
                roi = eye_roi

            cv2.imwrite(dst_path, roi)


train/drowsy:   0%|          | 0/22671 [00:00<?, ?it/s]c:\Users\USER\miniconda3\envs\skripsi_env\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
test/notdrowsy: 100%|██████████| 8237/8237 [01:16<00:00, 108.33it/s]


#  Code: simpan log

In [5]:
log_root = os.path.join(ROI_ROOT, "logs")
os.makedirs(log_root, exist_ok=True)

with open(os.path.join(log_root, "corrupted_files.txt"), "w") as f:
    for p in corrupted_files:
        f.write(p + "\n")

with open(os.path.join(log_root, "fallback_fullframe.txt"), "w") as f:
    for p in fallback_fullframe:
        f.write(p + "\n")

print(f"Total corrupted: {len(corrupted_files)}")
print(f"Total fallback full-frame: {len(fallback_fullframe)}")


Total corrupted: 0
Total fallback full-frame: 21740
